# Проект 4 — пункт C: читатели и писатели
Симуляция работы 8 читателей схемы зала и 2 продавцов билетов на 100 мест. Сравниваются режим приоритета читателей и справедливый режим, где ожидающий продавец временно закрывает вход новым читателям.
В конце выводятся продажи, число просмотров схемы, пропускная способность и максимальное ожидание продавца. Это учебная модель синхронизации потоков, а не подключение к реальной системе продажи билетов.

In [ ]:
"""Readers-writers simulation for Project 4, advanced task (C).

Eight readers inspect the seating chart while two sellers reserve seats.
Run with ``python booking_race.py`` to compare reader-priority and fair modes.
"""

from __future__ import annotations

import argparse
import threading
import time
from dataclasses import dataclass, field


SEAT_COUNT = 100
READER_COUNT = 8
SELLER_COUNT = 2


@dataclass
class Metrics:
    sold: int = 0
    views: int = 0
    seller_waits: list[float] = field(default_factory=list)
    lock: threading.Lock = field(default_factory=threading.Lock)


class ReadersWritersLock:
    """Multiple simultaneous readers, exclusive writers.

    In reader-priority mode new readers may pass waiting writers, which can
    starve sellers. Fair mode closes the reader gate as soon as a writer queues.
    """

    def __init__(self, reader_priority: bool) -> None:
        self.reader_priority = reader_priority
        self._condition = threading.Condition()
        self._active_readers = 0
        self._writer_active = False
        self._waiting_writers = 0

    def acquire_read(self) -> None:
        with self._condition:
            while self._writer_active or (
                not self.reader_priority and self._waiting_writers > 0
            ):
                self._condition.wait()
            self._active_readers += 1

    def release_read(self) -> None:
        with self._condition:
            self._active_readers -= 1
            if self._active_readers == 0:
                self._condition.notify_all()

    def acquire_write(self) -> None:
        with self._condition:
            self._waiting_writers += 1
            try:
                while self._writer_active or self._active_readers > 0:
                    self._condition.wait()
                self._writer_active = True
            finally:
                self._waiting_writers -= 1

    def release_write(self) -> None:
        with self._condition:
            self._writer_active = False
            self._condition.notify_all()


def run_simulation(
    *, reader_priority: bool, duration: float, read_time: float, sale_time: float
) -> Metrics:
    rw_lock = ReadersWritersLock(reader_priority)
    metrics = Metrics()
    stop_readers = threading.Event()
    start = time.perf_counter()

    def reader() -> None:
        while not stop_readers.is_set():
            rw_lock.acquire_read()
            try:
                # Readers only inspect the chart; they do not reserve seats.
                time.sleep(read_time)
                with metrics.lock:
                    metrics.views += 1
            finally:
                rw_lock.release_read()
            # Small pause prevents a single reader from monopolizing the CPU.
            time.sleep(0.0005)

    def seller() -> None:
        while True:
            requested_at = time.perf_counter()
            rw_lock.acquire_write()
            acquired_at = time.perf_counter()
            try:
                metrics.seller_waits.append(acquired_at - requested_at)
                with metrics.lock:
                    if metrics.sold >= SEAT_COUNT:
                        return
                    # Find and mark a free seat atomically while holding write lock.
                    metrics.sold += 1
                time.sleep(sale_time)
            finally:
                rw_lock.release_write()

    readers = [threading.Thread(target=reader, name=f"reader-{i + 1}") for i in range(READER_COUNT)]
    sellers = [threading.Thread(target=seller, name=f"seller-{i + 1}") for i in range(SELLER_COUNT)]

    for thread in readers + sellers:
        thread.start()

    # Let readers and sellers contend for the configured interval.
    time.sleep(duration)
    stop_readers.set()
    for thread in readers:
        thread.join()

    # Sellers finish selling all remaining seats after readers leave.
    for thread in sellers:
        thread.join()

    metrics.elapsed = time.perf_counter() - start  # type: ignore[attr-defined]
    return metrics


def report(label: str, metrics: Metrics) -> None:
    elapsed = metrics.elapsed  # type: ignore[attr-defined]
    maximum_wait = max(metrics.seller_waits, default=0.0)
    throughput = metrics.sold / elapsed if elapsed else 0.0
    print(f"{label}:")
    print(f"  Продано билетов: {metrics.sold}/{SEAT_COUNT}")
    print(f"  Просмотров схемы: {metrics.views}")
    print(f"  Время работы: {elapsed:.3f} с")
    print(f"  Пропускная способность: {throughput:.2f} билета/с")
    print(f"  Максимальное ожидание продавца: {maximum_wait:.3f} с")


def main() -> None:
    parser = argparse.ArgumentParser(description="Сравнение политик readers-writers")
    parser.add_argument("--duration", type=float, default=2.0,
                        help="длительность конкуренции читателей и продавцов (сек.)")
    parser.add_argument("--read-time", type=float, default=0.01,
                        help="время просмотра схемы (сек.)")
    parser.add_argument("--sale-time", type=float, default=0.005,
                        help="время оформления билета (сек.)")
    args = parser.parse_args()
    if min(args.duration, args.read_time, args.sale_time) <= 0:
        parser.error("все значения времени должны быть положительными")

    print("Проект 4, пункт C: 8 читателей и 2 продавца, 100 мест\n")
    reader_first = run_simulation(
        reader_priority=True, duration=args.duration,
        read_time=args.read_time, sale_time=args.sale_time,
    )
    fair = run_simulation(
        reader_priority=False, duration=args.duration,
        read_time=args.read_time, sale_time=args.sale_time,
    )
    report("Приоритет читателей", reader_first)
    print()
    report("Справедливый режим (продавец закрывает вход новым читателям)", fair)
    print("\nВывод: сравните максимальное ожидание продавца и число просмотренных схем.")


main()
